In [ ]:
import os
import pandas as pd

from typing import Optional

from datasets import Dataset, DatasetDict, load_dataset, load_from_disk, ClassLabel

from melanet.cache import CacheManager, FeatureExtractorCache, ClassifierCache
from metacentrum_utils import load_dataset_from_scratch, DATASET_SCRATCH_PREFIX

In [ ]:
IMAGE_ID_COLUMN_NAME = "isic_id"
LESION_ID_COLUMN_NAME = "lesion_id"
LABEL_COLUMN_NAME = "label"

In [ ]:
def load_and_validate_dataset(dataset_name: str, dataset_split: Optional[str] = None) -> Dataset:
    if os.path.exists(dataset_name):
        # Load from local path
        print(f"Loading dataset {dataset_name} ({dataset_split}) from local path")
        dataset = load_from_disk(
            dataset_path=dataset_name
        )
    elif dataset_name.startswith(DATASET_SCRATCH_PREFIX):
        # Load from scratch directory on Metacentrum
        print(f"Loading dataset {dataset_name} ({dataset_split}) from scratch storage")
        dataset = load_dataset_from_scratch(
            dataset_name
        )
    else:
        # Pull from HF or load it from cache
        print(f"Loading dataset {dataset_name} ({dataset_split}) from HF Hub")
        dataset = load_dataset(
            dataset_name,
            split=dataset_split
        )

    if not isinstance(dataset, Dataset):
        assert dataset_split is not None
        dataset = dataset[dataset_split]

    return dataset

In [ ]:
def create_pandas_features_dataset(
    cached_features_path: str
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    cached_features = CacheManager[FeatureExtractorCache].load(cached_features_path)
    cached_metadata = cached_features.metadata

    train_df = cached_features.cache["index"]
    validation_df = cached_features.cache["eval_dataset"]

    train_dataset = load_and_validate_dataset(
        cached_metadata["datasets"]["index"]["name"],
        cached_metadata["datasets"]["index"]["split"]
    )

    val_dataset = load_and_validate_dataset(
        cached_metadata["datasets"]["eval_dataset"]["name"],
        cached_metadata["datasets"]["eval_dataset"]["split"]
    )

    label_names = train_dataset.features[LABEL_COLUMN_NAME].names

    train_df[[LESION_ID_COLUMN_NAME, LABEL_COLUMN_NAME]] = train_df[IMAGE_ID_COLUMN_NAME].map({
        id: (lesion_id, label)
        for id, lesion_id, label in zip(
            train_dataset[IMAGE_ID_COLUMN_NAME],
            train_dataset[LESION_ID_COLUMN_NAME],
            train_dataset[LABEL_COLUMN_NAME]
        )
    }).to_list()

    validation_df[[LESION_ID_COLUMN_NAME, LABEL_COLUMN_NAME]] = validation_df[IMAGE_ID_COLUMN_NAME].map({
        id: (lesion_id, label)
        for id, lesion_id, label in zip(
            val_dataset[IMAGE_ID_COLUMN_NAME],
            val_dataset[LESION_ID_COLUMN_NAME],
            val_dataset[LABEL_COLUMN_NAME]
        )
    }).to_list()

    return train_df, validation_df, label_names

# Dataset from features

In [ ]:
cached_features_path = "/home/sulcm/datasets/milk10k/milk10k_DermLIP_features.pkl"
features_dataset_output_path = "/home/sulcm/datasets/milk10k/MILK10k_DermLIP_features"

In [ ]:
train_df, validation_df, label_names = create_pandas_features_dataset(cached_features_path)

In [ ]:
label_column = ClassLabel(
    num_classes=len(label_names),
    names=label_names
)

train_ds = Dataset.from_pandas(train_df).cast_column(
    LABEL_COLUMN_NAME,
    label_column
)

validation_ds = Dataset.from_pandas(validation_df).cast_column(
    LABEL_COLUMN_NAME,
    label_column
)

In [ ]:
features_dataset = DatasetDict({
    "train": train_ds,
    "validation": validation_ds
})
features_dataset

In [ ]:
# features_dataset.save_to_disk(features_dataset_output_path)

In [ ]:
lds = load_from_disk(
    "/home/sulcm/datasets/milk10k/MILK10k_DermLIP_features"
)
lds

# Merge two feature datasets

In [ ]:
cached_features_path_model_A = "/home/sulcm/datasets/milk10k/milk10k_vit_base_features.pkl"
cached_features_path_model_B = "/home/sulcm/datasets/milk10k/milk10k_DermLIP_features.pkl"
merged_features_dataset_output_path = "/home/sulcm/datasets/milk10k/MILK10k_ViT_base_DermLIP_features"

In [ ]:
train_df_A, validation_df_A, label_names_A = create_pandas_features_dataset(cached_features_path_model_A)

In [ ]:
train_df_A["melanet_features_ft_embeddings"][0].shape

In [ ]:
train_df_B, validation_df_B, label_names_B = create_pandas_features_dataset(cached_features_path_model_B)

In [ ]:
merge_on_cols = (IMAGE_ID_COLUMN_NAME, LESION_ID_COLUMN_NAME, LABEL_COLUMN_NAME)

assert label_names_A == label_names_B

merged_label_column = ClassLabel(
    num_classes=len(label_names_A),
    names=label_names_A
)

for check_col in merge_on_cols:
    assert train_df_A[check_col].equals(train_df_B[check_col])
    assert validation_df_A[check_col].equals(validation_df_B[check_col])

merged_train_df = pd.merge(train_df_A, train_df_B, how="inner", on=merge_on_cols)
merged_validation_df = pd.merge(validation_df_A, validation_df_B, how="inner", on=merge_on_cols)

In [ ]:
merged_train_ds = Dataset.from_pandas(merged_train_df).cast_column(
    LABEL_COLUMN_NAME,
    merged_label_column
)

merged_validation_ds = Dataset.from_pandas(merged_validation_df).cast_column(
    LABEL_COLUMN_NAME,
    merged_label_column
)

In [ ]:
merged_features_dataset = DatasetDict({
    "train": merged_train_ds,
    "validation": merged_validation_ds
})
merged_features_dataset

In [ ]:
# merged_features_dataset.save_to_disk(merged_features_dataset_output_path)

In [ ]:
lds = load_from_disk(
    "/home/sulcm/datasets/milk10k/MILK10k_ResNet50_DermLIP_features"
)
lds

# Dataset from logits

In [ ]:
cached_logits_path = ""
logits_dataset_output_path = ""

In [ ]:
cached_logits = CacheManager[ClassifierCache].load(cached_logits_path)